# Python SDK tour — `tensor` from CPython

*The named-axis differentiable tensor library reached for from Python instead of C++. Phase 6 SDK per [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md) + the [Phase 6 impl-plan](../../docs/impl-plans/2026-05-12_phase-6-python-sdk.md).*

This notebook is the Python-side companion to the C++ tutorials [`tutorials/00_intro.ipynb`](../../tutorials/00_intro.ipynb) and [`tutorials/01_formula-is-the-program.ipynb`](../../tutorials/01_formula-is-the-program.ipynb). Same domain, same vocabulary; one runs through the xcpp20 kernel, the other through CPython + nanobind.

> Educational, not production. Production users of named-tensor algebra in Python should stick with NumPy + jaxtyping / PyTorch named-tensors (still experimental). See [ADR-0001](../../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md).

## What this notebook covers

1. **Setup** — install + version check.
2. **`Axis` + `DynamicShape`** — the named-axis primitives.
3. **`DynamicTensor`** — float64 + float32 construction.
4. **Einstein-style broadcast** — `a_i + b_j → c_{ij}` from Python.
5. **NumPy interop** — `from_numpy` / `.numpy()` round-trip.
6. **Contraction** — `tensor.contract(W, x)` for matrix-vector and matrix-matrix.
7. **Cross-validation** — `tensor.contract` ≡ `np.einsum` for the canonical `ij,jk->ik` pattern.
8. **Where this fits** — phase-6 milestone map and cross-references.

Phase 6 status (2026-05-13): M1 (scaffold) ✅, M2 (DynamicTensor + arithmetic) ✅, M3 (contract + NumPy interop) ✅. M4 (autograd) / M5 (`_tex` Evaluator) / M6 (runtime backend selection + `0.2.0` PyPI publish) are next.

In [1]:
# Colab / Binder setup — install the tensor SDK on first run.
try:
    import tensor  # noqa: F401
except ImportError:
    import subprocess as _sp
    _sp.run(
        ["pip", "install", "--quiet",
         "git+https://github.com/uyuutosa/tensor.git"],
        check=True,
    )
    import tensor  # noqa: F401

## 1. Setup

From the repository root:

```bash
pip install -e . -v       # build the nanobind extension via scikit-build-core
pip install numpy pytest  # for the cross-validation cells below
```

Once 0.2.0 ships to PyPI (Phase 6 M6 exit per the impl-plan), end users will install via:

```bash
pip install tensor-named-axis
# or
conda install -c conda-forge tensor-named-axis
```

In [2]:
import numpy as np
import tensor

print(f"tensor.__version__: {tensor.__version__}")
print(f"hello: {tensor.hello()}")

tensor.__version__: 0.1.0+dev
hello: hello from tensor::core


## 2. Named axes — `Axis` and `DynamicShape`

Every tensor in this library carries explicit axis labels. The label is what makes `a_i + b_j` mean a *5×5 outer-sum table*, not an error.

An `Axis` is a `(label, extent)` pair. A `DynamicShape` is an ordered list of axes.

In [3]:
i_axis = tensor.Axis("i", 5)
j_axis = tensor.Axis("j", 3)
print(f"i: {i_axis}")
print(f"j: {j_axis}")

shape = tensor.DynamicShape([i_axis, j_axis])
print(f"shape: {shape}")
print(f"rank = {shape.rank()}, total = {shape.total()}")

i: Axis("i", 5)
j: Axis("j", 3)
shape: DynamicShape([Axis("i", 5), Axis("j", 3)])
rank = 2, total = 15


## 3. `DynamicTensor` — the runtime tensor type

`DynamicTensor` is the float64 (educational default) named-axis tensor. `DynamicTensorF32` is its float32 twin — used when interoperating with the WebGPU backend (which is float-only per [ADR-0012](../../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md)).

Construct one from a `DynamicShape` plus a flat Python list of values; the length must equal `shape.total()`. The `__str__` / `__repr__` use the project's 2016-descended ASCII pretty-print.

In [4]:
a = tensor.DynamicTensor(
    tensor.DynamicShape([tensor.Axis("i", 5)]),
    [1.0, 2.0, 3.0, 4.0, 5.0],
)
print("a =")
print(a)

b = tensor.DynamicTensor(
    tensor.DynamicShape([tensor.Axis("j", 5)]),
    [1.0, 2.0, 3.0, 4.0, 5.0],
)
print("\nb =")
print(b)

a =
-*-Infos-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Total size  : 5
Shape       : (i: 5)
Num. of Dim.: 1
-*-Values-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
[ 1, 2, 3, 4, 5 ]


b =
-*-Infos-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Total size  : 5
Shape       : (j: 5)
Num. of Dim.: 1
-*-Values-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
[ 1, 2, 3, 4, 5 ]



## 4. Einstein-style broadcast

`a + b` between two tensors with **distinct** axis labels produces the outer sum — `c_{ij} = a_i + b_j`. With **shared** labels the axes collapse pairwise to element-wise. This matches the C++ side's behaviour exactly (cross-validated in `tests/test_ops.cpp`).

In [5]:
# Distinct labels → 5x5 outer-sum table
c = a + b
print("c = a + b (c_{ij} = a_i + b_j):")
print(c)
print(f"rank = {c.shape.rank()}, total = {c.shape.total()}")

# The four corners reproduce the C++ test_ops.cpp values:
#   c[0*5 + 0] = 1 + 1 = 2
#   c[0*5 + 4] = 1 + 5 = 6
#   c[4*5 + 0] = 5 + 1 = 6
#   c[4*5 + 4] = 5 + 5 = 10
assert c[0] == 2.0 and c[4] == 6.0 and c[20] == 6.0 and c[24] == 10.0
print("\nCross-validation OK — four corners match C++ tests/test_ops.cpp.")

c = a + b (c_{ij} = a_i + b_j):
-*-Infos-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Total size  : 25
Shape       : (i: 5, j: 5)
Num. of Dim.: 2
-*-Values-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
[[ 2, 3, 4, 5, 6 ]
 [ 3, 4, 5, 6, 7 ]
 [ 4, 5, 6, 7, 8 ]
 [ 5, 6, 7, 8, 9 ]
 [ 6, 7, 8, 9, 10 ]]

rank = 2, total = 25

Cross-validation OK — four corners match C++ tests/test_ops.cpp.


In [6]:
# Same labels → element-wise rank-1 (no outer expansion)
a_again = tensor.DynamicTensor(
    tensor.DynamicShape([tensor.Axis("i", 5)]),
    [10.0, 20.0, 30.0, 40.0, 50.0],
)
elementwise = a + a_again
print("a + a_again (both labelled 'i'):")
print(elementwise)
assert elementwise.shape.rank() == 1
assert elementwise[0] == 11.0 and elementwise[4] == 55.0

a + a_again (both labelled 'i'):
-*-Infos-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Total size  : 5
Shape       : (i: 5)
Num. of Dim.: 1
-*-Values-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
[ 11, 22, 33, 44, 55 ]



## 5. NumPy interop

`tensor.from_numpy(arr, labels)` wraps a contiguous NumPy array as a `DynamicTensor`, naming each dimension explicitly. `.numpy()` is the round-trip — returns a fresh Python-owned `numpy.ndarray` with the tensor's shape.

Data is **copied** on both directions in M3; the two objects' lifetimes are independent.

In [7]:
arr = np.arange(6, dtype=np.float64).reshape(2, 3)
print("arr =\n", arr)

t = tensor.from_numpy(arr, ["i", "j"])
print("\nas a named-axis tensor t (labels = ('i','j')):")
print(t)

back = t.numpy()
print("\nround-trip back =\n", back)

np.testing.assert_array_equal(back, arr)
print("\nRound-trip is bit-exact.")

arr =
 [[0. 1. 2.]
 [3. 4. 5.]]

as a named-axis tensor t (labels = ('i','j')):
-*-Infos-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Total size  : 6
Shape       : (i: 2, j: 3)
Num. of Dim.: 2
-*-Values-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
[[ 0, 1, 2 ]
 [ 3, 4, 5 ]]


round-trip back =
 [[0. 1. 2.]
 [3. 4. 5.]]

Round-trip is bit-exact.


## 6. Contraction — `tensor.contract`

`tensor.contract(a, b)` is the named-axis Einstein-sum primitive: axes that appear in both inputs are summed over; axes that appear in only one form the result. Matrix-matrix, matrix-vector, outer product, and inner product are all the same op specialised by the label structure.

Below: a matrix-vector product `y_i = Σ_j W_{ij} x_j`, then a matrix-matrix product `C_{ik} = Σ_j A_{ij} B_{jk}`. Cross-validated against `tests/test_contract.cpp`.

In [8]:
# Matvec: y_i = Σ_j W_{ij} x_j
W = tensor.from_numpy(np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]), ["i", "j"])
x = tensor.from_numpy(np.array([10.0, 20.0, 30.0]), ["j"])
y = tensor.contract(W, x)
print("y = W · x (named-axis matvec):\n", y.numpy())
# Expected: [1*10 + 2*20 + 3*30, 4*10 + 5*20 + 6*30] = [140, 320]
assert y.numpy().tolist() == [140.0, 320.0]

y = W · x (named-axis matvec):
 [140. 320.]


In [9]:
# Matmul: C_{ik} = Σ_j A_{ij} B_{jk}
A_np = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
B_np = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
A = tensor.from_numpy(A_np, ["i", "j"])
B = tensor.from_numpy(B_np, ["j", "k"])
C = tensor.contract(A, B)
print("C = A · B (named-axis matmul):\n", C.numpy())
# Cross-validate against NumPy
np.testing.assert_array_equal(C.numpy(), A_np @ B_np)

C = A · B (named-axis matmul):
 [[ 4.  5.]
 [10. 11.]]


## 7. Cross-validation with `np.einsum`

The exit criterion for [Phase 6 M3](../../docs/impl-plans/2026-05-12_phase-6-python-sdk.md): `tensor.contract` must agree with `np.einsum` to `1e-12` for `float64` on the canonical matrix-matrix pattern `ij,jk->ik`. Confirm it on a random sample.

In [10]:
rng = np.random.default_rng(seed=42)
A_np = rng.standard_normal((4, 7))
B_np = rng.standard_normal((7, 5))

A_t = tensor.from_numpy(A_np, ["i", "j"])
B_t = tensor.from_numpy(B_np, ["j", "k"])
C_t = tensor.contract(A_t, B_t).numpy()
C_einsum = np.einsum("ij,jk->ik", A_np, B_np)

assert C_t.shape == C_einsum.shape
max_err = float(np.max(np.abs(C_t - C_einsum)))
print(f"max abs error: {max_err:.2e}")
np.testing.assert_allclose(C_t, C_einsum, rtol=1e-12, atol=1e-12)
print("tensor.contract == np.einsum within 1e-12 — exit criterion met.")

max abs error: 0.00e+00
tensor.contract == np.einsum within 1e-12 — exit criterion met.


## 8. Where this fits

**Architecture**: the Python SDK is a DrivingAdapter ([ADR-0009](../../docs/arc42/09-decisions/0009-adopt-ddd-ubiquitous-language-and-hexagonal-lite.md) + [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md); see [arc42 §5](../../docs/arc42/05-building-blocks/overview.md)). It consumes the same C++ Domain that the xcpp20 tutorials and the C++ test suite exercise. The named-axis vocabulary is one — the same `Axis` lives in both surfaces.

**Phase 6 milestone map**:

| Milestone | Surface | Status (2026-05-13) |
| --------- | ------- | -------------------- |
| P6.M1 | scaffold + smoke binding | ✅ shipped |
| P6.M2 | `Axis` + `DynamicShape` + `DynamicTensor` + 4 arithmetic ops | ✅ shipped |
| **P6.M3** | **`contract` + NumPy interop** | **✅ shipped — this notebook covers it** |
| P6.M4 | autograd (`DynamicVariable`, `backward`, `gradient_check`) | next |
| P6.M5 | `tex.parse` + `Evaluator` (the `_tex` UDL equivalent) | after M4 |
| P6.M6 | runtime backend selection + `0.2.0` release | exit |

**Sibling notebooks**:

- [`tutorials/00_intro.ipynb`](../../tutorials/00_intro.ipynb) — the C++ side of the same content (named-axis fundamentals).
- [`tutorials/01_formula-is-the-program.ipynb`](../../tutorials/01_formula-is-the-program.ipynb) — the `_tex` UDL end-to-end, which gets a Python counterpart in P6.M5.
- [`tutorials/05_autograd-from-scratch.ipynb`](../../tutorials/05_autograd-from-scratch.ipynb) — tape-based autograd in C++, mirrored in P6.M4.
- [`tutorials/08_swappable-backends.ipynb`](../../tutorials/08_swappable-backends.ipynb) — the Hexagonal-lite payoff; P6.M6 exposes the same backend choice from Python.

**Cross-references**:

- [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md) — the architectural decisions behind this SDK.
- [Phase 6 impl-plan](../../docs/impl-plans/2026-05-12_phase-6-python-sdk.md) — the six-milestone breakdown.
- [arc42 §12 Glossary](../../docs/arc42/12-glossary/overview.md) — the named-axis vocabulary; every Python symbol shown here has a glossary entry.